# 代码评分评估：分类任务

在本节课中，我们将从头开始实现一个稍微复杂的代码评分评估，用于测试客户投诉分类提示词。我们的目标是编写一个能够可靠地将客户投诉分类到以下类别的提示词：

* 软件缺陷
* 硬件故障
* 用户错误
* 功能请求
* 服务中断

例如，以下投诉文本：

> 网站完全无法访问，我打不开任何页面

应被分类为 `服务中断`

在某些情况下，我们可能允许最多两个适用的分类类别，例如：

> 我觉得我安装什么东西的时候操作错了，现在电脑完全开不了机

这应被同时分类为 `用户错误` 和 `硬件故障`

---


## 评估数据集

我们首先定义评估数据集的输入和标准答案。请记住，通常我们需要大约 100 个输入的评估数据集，但为了保持课程简单（并且运行快速、经济实惠），我们使用了一个精简版。

这个测试集由字典列表组成，每个字典包含 `complaint` 和 `golden_answer` 键：

In [2]:
eval_data = [
    {
        "complaint": "The app crashes every time I try to upload a photo",
        "golden_answer": ["Software Bug"]
    },
    {
        "complaint": "My printer isn't recognized by my computer",
        "golden_answer": ["Hardware Malfunction"]
    },
    {
        "complaint": "I can't figure out how to change my password",
        "golden_answer": ["User Error"]
    },
    {
        "complaint": "The website is completely down, I can't access any pages",
        "golden_answer": ["Service Outage"]
    },
    {
        "complaint": "It would be great if the app had a dark mode option",
        "golden_answer": ["Feature Request"]
    },
    {
        "complaint": "The software keeps freezing when I try to save large files",
        "golden_answer": ["Software Bug"]
    },
    {
        "complaint": "My wireless mouse isn't working, even with new batteries",
        "golden_answer": ["Hardware Malfunction"]
    },
    {
        "complaint": "I accidentally deleted some important files, can you help me recover them?",
        "golden_answer": ["User Error"]
    },
    {
        "complaint": "None of your servers are responding, is there an outage?",
        "golden_answer": ["Service Outage"]
    },
    {
        "complaint": "Could you add a feature to export data in CSV format?",
        "golden_answer": ["Feature Request"]
    },
    {
        "complaint": "The app is crashing and my phone is overheating",
        "golden_answer": ["Software Bug", "Hardware Malfunction"]
    },
    {
        "complaint": "I can't remember my password!",
        "golden_answer": ["User Error"]
    },
    {
        "complaint": "The new update broke something and the app no longer works for me",
        "golden_answer": ["Software Bug"]
    },
    {
        "complaint": "I think I installed something incorrectly, now my computer won't start at all",
        "golden_answer": ["User Error", "Hardware Malfunction"]
    },
    {
        "complaint": "Your service is down, and I urgently need a feature to batch process files",
        "golden_answer": ["Service Outage", "Feature Request"]
    },
    {
        "complaint": "The graphics card is making weird noises",
        "golden_answer": ["Hardware Malfunction"]
    },
    {
        "complaint": "My keyboard just totally stopped working out of nowhere",
        "golden_answer": ["Hardware Malfunction"]
    },
    {
        "complaint": "Whenever I open your app, my phone gets really slow",
        "golden_answer": ["Software Bug"]
    },
    {
        "complaint": "Can you make the interface more user-friendly? I always get lost in the menus",
        "golden_answer": ["Feature Request", "User Error"]
    },
    {
        "complaint": "The cloud storage isn't syncing and I can't access my files from other devices",
        "golden_answer": ["Software Bug", "Service Outage"]
    }
]

---

## 初始提示词

我们从基本提示词开始，测量它的表现。下面这个提示词生成函数接受一个 `complaint` 作为参数，并返回一个提示词字符串：

In [3]:
def basic_prompt(complaint):
    return f"""
    Classify the following customer complaint into one or more of these categories: 
    Software Bug, Hardware Malfunction, User Error, Feature Request, or Service Outage.
    Only respond with the matching category or categories and nothing else.

    Complaint: {complaint}

    Classification:
    """

---

## 收集输出

接下来，我们编写评估提示词的逻辑。这个逻辑比上一节课的"数腿"示例要复杂一些：

In [4]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

def get_model_response(prompt, model_name):
    response = client.messages.create(
        model=model_name,
        max_tokens=200,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response.content[0].text

def calculate_accuracy(eval_data, model_responses):
    correct_predictions = 0
    total_predictions = len(eval_data)
    
    for item, response in zip(eval_data, model_responses):
        golden_set = set(category.lower() for category in item["golden_answer"])
        prediction_set = set(category.strip().lower() for category in response.split(','))
        
        if golden_set == prediction_set:
            correct_predictions += 1
    
    return correct_predictions / total_predictions

def evaluate_prompt(prompt_func, eval_data, model_name):
    print(f"Evaluating with model: {model_name}")
    model_responses = [get_model_response(prompt_func(item['complaint']), model_name) for item in eval_data]
    accuracy = calculate_accuracy(eval_data, model_responses)
    
    print(f"Accuracy: {accuracy:.2%}")
    
    for item, response in zip(eval_data, model_responses):
        print(f"\nComplaint: {item['complaint']}")
        print(f"Golden Answer: {item['golden_answer']}")
        print(f"Model Response: {response}")
    return accuracy

`evaluate_prompt` 函数执行以下步骤：

1. 它将每个输入传递给我们的提示词生成函数，然后使用 `get_model_response` 函数将生成的提示词传递给模型，收集返回的响应。
2. 它通过调用 `calculate_accuracy` 函数将模型输出答案与数据集中的标准答案进行比较来计算准确率。
3. `calculate_accuracy` 函数使用 `集合` 检查每个模型输出中是否存在适当的分类类别。请记住，这与之前的"数腿"评估不同，这不是精确匹配评估。
4. `calculate_accuracy` 返回一个准确率分数
5. `evaluate_prompt` 打印最终结果

**请注意，这次我们的评分逻辑不是通过精确字符串匹配来评分，而是使用 `集合` 来检查模型输出中是否存在相应值。**

让我们用初始的 `basic_prompt` 测试一下

In [5]:
evaluate_prompt(basic_prompt, eval_data, model_name="claude-3-haiku-20240307")

Evaluating with model: claude-3-haiku-20240307
Accuracy: 85.00%

Complaint: The app crashes every time I try to upload a photo
Golden Answer: ['Software Bug']
Model Response: Software Bug

Complaint: My printer isn't recognized by my computer
Golden Answer: ['Hardware Malfunction']
Model Response: Hardware Malfunction

Complaint: I can't figure out how to change my password
Golden Answer: ['User Error']
Model Response: User Error

Complaint: The website is completely down, I can't access any pages
Golden Answer: ['Service Outage']
Model Response: Service Outage

Complaint: It would be great if the app had a dark mode option
Golden Answer: ['Feature Request']
Model Response: Feature Request

Complaint: The software keeps freezing when I try to save large files
Golden Answer: ['Software Bug']
Model Response: Software Bug

Complaint: My wireless mouse isn't working, even with new batteries
Golden Answer: ['Hardware Malfunction']
Model Response: Hardware Malfunction

Complaint: I accidenta

0.85

---

## 改进后的提示词
我们最初的提示词得到了 85% 的准确率。让我们对提示词做一些修改，重新运行评估，希望获得更好的分数。

以下提示词包含了对类别的详细解释，以及 9 个输入输出示例对：

In [6]:
def improved_prompt(complaint):
    return f"""
    You are an AI assistant specializing in customer support issue classification. Your task is to analyze customer complaints and categorize them into one or more of the following categories:

    1. Software Bug: Issues related to software not functioning as intended.
    2. Hardware Malfunction: Problems with physical devices or components.
    3. User Error: Difficulties arising from user misunderstanding or misuse.
    4. Feature Request: Suggestions for new functionalities or improvements.
    5. Service Outage: System-wide issues affecting service availability.

    Important Guidelines:
    - A complaint may fall into multiple categories. If so, list all that apply but try to prioritize picking a single category when possible.

    Examples:
    1. Complaint: "The app crashes when I try to save my progress."
    Classification: Software Bug

    2. Complaint: "My keyboard isn't working after I spilled coffee on it."
    Classification: Hardware Malfunction

    3. Complaint: "I can't find the login button on your website."
    Classification: User Error

    4. Complaint: "It would be great if your app had a dark mode."
    Classification: Feature Request

    5. Complaint: "None of your services are loading for me or my colleagues."
    Classification: Service Outage

    6. Complaint "Complaint: The app breaks every time I try to change my profile picture"
    Classification: Software Bug

    7. Complaint "The app is acting buggy on my phone and it seems like your website is down, so I'm completely stuck!"
    Classification: Software Bug, Service Outage

    8. Complaint: "Your software makes my computer super laggy and awful, I hate it!"
    Classification: Software Bug

    9. Complaint: "Your dumb app always breaks when I try to do anything with images."
    Classification: 'Software Bug'

    Now, please classify the following customer complaint:

    <complaint>{complaint}</complaint>

    Only respond with the appropriate categories and nothing else.
    Classification:
    """

让我们用改进后的提示词运行评估：

In [80]:
evaluate_prompt(improved_prompt, eval_data, model_name="claude-3-haiku-20240307")

Evaluating with model: claude-3-haiku-20240307
Accuracy: 100.00%

Complaint: The app crashes every time I try to upload a photo
Golden Answer: ['Software Bug']
Model Response: Software Bug

Complaint: My printer isn't recognized by my computer
Golden Answer: ['Hardware Malfunction']
Model Response: Hardware Malfunction

Complaint: I can't figure out how to change my password
Golden Answer: ['User Error']
Model Response: User Error

Complaint: The website is completely down, I can't access any pages
Golden Answer: ['Service Outage']
Model Response: Service Outage

Complaint: It would be great if the app had a dark mode option
Golden Answer: ['Feature Request']
Model Response: Feature Request

Complaint: The software keeps freezing when I try to save large files
Golden Answer: ['Software Bug']
Model Response: Software Bug

Complaint: My wireless mouse isn't working, even with new batteries
Golden Answer: ['Hardware Malfunction']
Model Response: Hardware Malfunction

Complaint: I accident

1.0

使用更新、改进后的提示词，我们获得了 100% 的准确率！

我们再次遵循此图表中概述的标准提示词 + 评估循环：



**请记住，这是一个非常简单的评估，使用的是非常小的数据集。本课程的目的是说明代码评分评估的一般流程，而不是作为生产级评估的典型示例！**

这种方法可行，但从头开始编写所有评估逻辑有点费力，而且很难并排比较结果。如果我们使用一个能生成格式良好结果的工具，包含图表和图形，并且可以轻松地在多个模型上运行评估，那会怎么样？在下一节课中，我们将看到这样的工具！接下来，我们将了解一个评估框架，它可以轻松编写可重复、可扩展的评估，用于生产环境用例。